# LLM API Platform — Google Colab Model Worker
### Model: Qwen2.5-7B-Instruct (AWQ 4-bit) | Inference Engine: vLLM | Network: Cloudflare Tunnel

This notebook runs **vLLM** serving **Qwen2.5-7B-Instruct** on Google Colab's GPU (Tesla T4, L4, or A100) and connects it directly to the Central API Hosting Platform via an automated Cloudflare Tunnel and heartbeat registration loop.

**Instructions:**
1. Ensure GPU acceleration is enabled: `Runtime` -> `Change runtime type` -> select `T4 GPU` (or L4/A100 if on Pro).
2. Set your `PLATFORM_URL` and `WORKER_SECRET` below.
3. Click `Runtime` -> `Run all`.

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

In [ ]:
# Step 2: Install vLLM and Cloudflare Tunnel
!pip install -q --upgrade pip
!pip install -q vllm requests

import os, subprocess
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    print("cloudflared installed successfully!")

In [ ]:
# Step 3: Worker Configuration
import uuid

# Set your deployed Central Platform domain (e.g., https://api.yourdomain.com)
# If running locally via ngrok/localtunnel, put that public URL here
PLATFORM_URL = "https://your-platform-domain.com"

# Must match WORKER_SECRET environment variable on your platform
WORKER_SECRET = "colab-worker-secret-key-123"

# 4-bit AWQ fits comfortably on Google Colab's 15GB Tesla T4 GPU
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_ALIAS = "qwen2.5-7b"
WORKER_ID = f"colab-worker-{uuid.uuid4().hex[:8]}"

print(f"Worker ID: {WORKER_ID}")
print(f"Target Platform: {PLATFORM_URL}")
print(f"Model: {MODEL_ID}")

In [ ]:
# Step 4: Launch vLLM in Background
import subprocess, time, requests

print("Starting vLLM inference server in background...")
vllm_cmd = [
    "vllm", "serve", MODEL_ID,
    "--port", "8000",
    "--dtype", "float16",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "8192",
    "--served-model-name", MODEL_ALIAS
]

vllm_process = subprocess.Popen(vllm_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Waiting for vLLM to download model weights and initialize KV-cache...")
print("(This takes approximately 2-4 minutes on first load)")

start_wait = time.time()
vllm_ready = False
while time.time() - start_wait < 600:
    try:
        r = requests.get("http://127.0.0.1:8000/v1/models", timeout=2)
        if r.status_code == 200 and MODEL_ALIAS in r.text:
            vllm_ready = True
            print(f"\u2705 vLLM is ONLINE and ready! (Took {int(time.time() - start_wait)}s)")
            break
    except Exception:
        pass
    time.sleep(5)

if not vllm_ready:
    raise RuntimeError("Timed out waiting for vLLM to start. Check GPU memory with !nvidia-smi")

In [ ]:
# Step 5: Start Cloudflare Tunnel and Obtain Public Endpoint
import re, time

print("Establishing Cloudflare Quick Tunnel on port 8000...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stderr=subprocess.PIPE, text=True
)

public_tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if not public_tunnel_url:
    raise RuntimeError("Failed to obtain Cloudflare tunnel URL.")

print(f"\u2705 Public Tunnel Active: {public_tunnel_url}")

In [ ]:
# Step 6: Auto-Register with Central Platform & Start Heartbeat Loop
import atexit, sys

reg_payload = {
    "worker_id": WORKER_ID,
    "model_name": MODEL_ALIAS,
    "endpoint_url": public_tunnel_url,
    "secret": WORKER_SECRET,
    "gpu_info": "Tesla T4 GPU (Google Colab)"
}

print(f"Registering with platform at {PLATFORM_URL}/api/worker/register...")
try:
    reg_resp = requests.post(f"{PLATFORM_URL}/api/worker/register", json=reg_payload, timeout=10)
    print(f"Registration response: {reg_resp.status_code} - {reg_resp.text}")
except Exception as e:
    print(f"Warning: Could not connect to platform: {e}. Check PLATFORM_URL.")

def deregister():
    print("Deregistering worker from platform...")
    try:
        requests.post(f"{PLATFORM_URL}/api/worker/deregister", json={
            "worker_id": WORKER_ID,
            "secret": WORKER_SECRET
        }, timeout=5)
    except Exception:
        pass
    try:
        vllm_process.terminate()
        tunnel_proc.terminate()
    except Exception:
        pass

atexit.register(deregister)

print("\n\U0001f680 Worker is fully active! Emitting heartbeats every 15 seconds...")
print("(Press Stop button in notebook to gracefully shut down the worker)\n")

try:
    heartbeat_count = 0
    while True:
        # Check local vLLM health
        try:
            h = requests.get("http://127.0.0.1:8000/v1/models", timeout=3)
            if h.status_code != 200:
                print("Local vLLM health check failed!")
                break
        except Exception:
            print("Cannot contact local vLLM! Exiting...")
            break

        # Send heartbeat to central platform
        try:
            hb_resp = requests.post(f"{PLATFORM_URL}/api/worker/heartbeat", json={
                "worker_id": WORKER_ID,
                "secret": WORKER_SECRET
            }, timeout=5)
            heartbeat_count += 1
            if heartbeat_count % 4 == 0:  # Print every 1 min
                print(f"[Heartbeat] Heartbeat #{heartbeat_count} confirmed (Status: healthy)")
        except Exception as e:
            print(f"Heartbeat ping failed: {e}")

        time.sleep(15)
except KeyboardInterrupt:
    print("\nNotebook stopped by user. Cleaning up...")
    deregister()
    print("Worker offline.")